# 05 PSO Hyperparameter Tuning (All Datasets)

This notebook tunes CNN hyperparameters with PSO on **all three datasets** (Desharnais, COCOMO-81, China).

**Key features:**
- Enhanced CNN with BatchNormalization, Dropout, multi-layer support
- EarlyStopping + ReduceLROnPlateau callbacks
- Log-transform on effort target for skew correction
- 6D PSO: [filters, kernel_size, dense_units, lr, dropout_rate, num_conv_layers]
- Per-dataset models saved to `models/` for web app

In [ ]:
import json
import random
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")

root_dir = Path.cwd()
if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
    root_dir = root_dir.parent

if not (root_dir / "src").exists():
    raise FileNotFoundError("Could not find project root containing 'src' directory")

sys.path.insert(0, str(root_dir))

from src.cnn_model import build_cnn_regressor, reshape_for_cnn, train_cnn_model
from src.cv_pipeline import cross_validate_cnn, cross_validate_estimator
from src.evaluate import evaluate_predictions, save_metrics
from src.feature_engineering import inverse_log_transform, log_transform_target
from src.pso_optimizer import build_cnn_pso_objective, decode_cnn_hyperparameters, get_cnn_pso_bounds, tune_cnn_with_pso

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("xgboost not installed — XGBoost rows will be skipped in the final comparison")

processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"
models_dir = root_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Load all 3 datasets with display names for the final standardized comparison table.
dataset_files = {
    "China": {"key": "china", "file": "china_processed.csv"},
    "COCOMO81": {"key": "cocomo81", "file": "cocomo81_processed.csv"},
    "Desharnais": {"key": "desharnais", "file": "desharnais_processed.csv"},
}

datasets = {}
for dataset_name, dataset_info in dataset_files.items():
    path = processed_dir / dataset_info["file"]
    if path.exists():
        datasets[dataset_name] = {
            "key": dataset_info["key"],
            "data": pd.read_csv(path),
        }
        print(f"Loaded {dataset_name}: {datasets[dataset_name]['data'].shape}")

print(f"\n{len(datasets)} datasets ready for PSO tuning and final comparison")

Loaded china: (499, 19)
Loaded desharnais: (81, 13)
Loaded cocomo81: (63, 20)

3 datasets ready for PSO tuning


In [ ]:
# --- Helper: prepare leakage-free splits and model builders ---

def get_baseline_model_builders():
    """Return fresh model builders for the standardized final comparison."""
    model_builders = {
        "LinearRegression": lambda: LinearRegression(),
        "RandomForest": lambda: RandomForestRegressor(
            n_estimators=300, random_state=SEED, n_jobs=-1
        ),
    }
    if HAS_XGBOOST:
        model_builders["XGBoost"] = lambda: XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=SEED,
        )
    return model_builders


def prepare_dataset(df, use_log_transform=True):
    """Create 60/20/20 train/validation/test splits for leakage-free tuning."""
    target_col = df.columns[-1]
    X = df.drop(columns=[target_col]).values.astype(np.float32)
    y = df[target_col].values.astype(np.float32)

    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=0.25, random_state=SEED
    )

    # Keep X_test/y_test untouched until the final evaluation step.
    y_train_fit = log_transform_target(y_train, use_log_transform=use_log_transform).astype(np.float32)
    y_val_fit = log_transform_target(y_val, use_log_transform=use_log_transform).astype(np.float32)

    return {
        "X_full": X,
        "y_full": y,
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "y_train_fit": y_train_fit,
        "y_val_fit": y_val_fit,
        "X_train_cnn": reshape_for_cnn(X_train),
        "X_val_cnn": reshape_for_cnn(X_val),
        "X_test_cnn": reshape_for_cnn(X_test),
        "n_features": X.shape[1],
    }


def make_cnn_builder(input_length, hyperparameters=None):
    """Return a zero-argument model builder for CV and final training."""
    hyperparameters = hyperparameters or {}
    return lambda: build_cnn_regressor(
        input_length=input_length,
        filters=int(hyperparameters.get("filters", 32)),
        kernel_size=int(hyperparameters.get("kernel_size", 3)),
        dense_units=int(hyperparameters.get("dense_units", 64)),
        learning_rate=float(hyperparameters.get("learning_rate", 1e-3)),
        dropout_rate=float(hyperparameters.get("dropout_rate", 0.2)),
        num_conv_layers=int(hyperparameters.get("num_conv_layers", 1)),
    )


def train_baseline_cnn(splits, use_log_transform=True):
    """Train the baseline CNN on the train split and validate on the val split."""
    model = make_cnn_builder(splits["n_features"])()
    model, history = train_cnn_model(
        model,
        splits["X_train_cnn"], splits["y_train_fit"],
        splits["X_val_cnn"], splits["y_val_fit"],
        epochs=100, batch_size=32, verbose=0, use_callbacks=True,
    )
    preds_fit = model.predict(splits["X_test_cnn"], verbose=0).ravel()
    preds = inverse_log_transform(preds_fit, use_log_transform=use_log_transform)
    metrics = evaluate_predictions("cnn_baseline", splits["y_test"], preds)
    return model, history, metrics


def train_pso_cnn(splits, use_log_transform=True):
    """Tune with PSO on train/val only, then evaluate once on the untouched test split."""
    lower_bounds, upper_bounds = get_cnn_pso_bounds()
    objective_fn = build_cnn_pso_objective(
        X_train=splits["X_train"],
        y_train=splits["y_train"],
        X_val=splits["X_val"],
        y_val=splits["y_val"],
        input_length=splits["n_features"],
        epochs=30,
        use_log_transform=use_log_transform,
        verbose=0,
    )
    best_cost, best_position = tune_cnn_with_pso(
        objective_fn=objective_fn,
        dimensions=6,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
        n_particles=15,
        iters=25,
    )
    best_hyperparameters = decode_cnn_hyperparameters(best_position)

    model = make_cnn_builder(
        splits["n_features"],
        hyperparameters=best_hyperparameters,
    )()
    model, history = train_cnn_model(
        model,
        splits["X_train_cnn"], splits["y_train_fit"],
        splits["X_val_cnn"], splits["y_val_fit"],
        epochs=100,
        batch_size=int(best_hyperparameters.get("batch_size", 32)),
        verbose=0,
        use_callbacks=True,
    )
    preds_fit = model.predict(splits["X_test_cnn"], verbose=0).ravel()
    preds = inverse_log_transform(preds_fit, use_log_transform=use_log_transform)
    metrics = evaluate_predictions("cnn_pso", splits["y_test"], preds)
    return model, history, metrics, best_hyperparameters, best_cost


print("Helper functions defined.")

Helper functions defined.


In [ ]:
# --- Run leakage-free holdout training and 5-fold CV on ALL datasets ---

all_cnn_results = []
all_histories = {}
all_hyperparams = {}
final_rows = []

for dataset_name, dataset_info in datasets.items():
    dataset_key = dataset_info["key"]
    df = dataset_info["data"]

    print(f"\n{'='*70}")
    print(f"  Dataset: {dataset_name} ({len(df)} samples, {df.shape[1]-1} features)")
    print(f"{'='*70}")

    splits = prepare_dataset(df, use_log_transform=True)

    # 1. Baseline CNN holdout run
    print("\n[1/4] Training baseline CNN with a train/validation split...")
    baseline_model, baseline_hist, baseline_metrics = train_baseline_cnn(splits)
    baseline_metrics["dataset"] = dataset_key
    all_cnn_results.append(baseline_metrics)
    print(f"  Baseline CNN — MAE: {baseline_metrics['mae'].values[0]:.2f}, "
          f"RMSE: {baseline_metrics['rmse'].values[0]:.2f}, "
          f"R²: {baseline_metrics['r2'].values[0]:.4f}")

    # 2. PSO tuning uses only train/validation data; the test split stays untouched here.
    print("\n[2/4] Running PSO on the train/validation split only...")
    pso_model, pso_hist, pso_metrics, best_hyperparameters, best_cost = train_pso_cnn(
        splits,
        use_log_transform=True,
    )
    pso_metrics["dataset"] = dataset_key
    all_cnn_results.append(pso_metrics)
    all_hyperparams[dataset_key] = best_hyperparameters

    print(f"  Best PSO validation RMSE: {best_cost:.4f}")
    print(f"  Hyperparams: {best_hyperparameters}")
    print(f"  CNN+PSO    — MAE: {pso_metrics['mae'].values[0]:.2f}, "
          f"RMSE: {pso_metrics['rmse'].values[0]:.2f}, "
          f"R²: {pso_metrics['r2'].values[0]:.4f}")

    baseline_model.save(models_dir / f"cnn_baseline_{dataset_key}.h5")
    pso_model.save(models_dir / f"cnn_pso_{dataset_key}.h5")

    all_histories[dataset_key] = {
        "baseline": {k: [float(v) for v in values] for k, values in baseline_hist.items()},
        "pso": {k: [float(v) for v in values] for k, values in pso_hist.items()},
    }

    # 3. Cross-validate the classical baselines with the same log-target workflow.
    print("\n[3/4] Running 5-fold CV for classical baselines...")
    for model_name, model_builder in get_baseline_model_builders().items():
        cv_summary = cross_validate_estimator(
            model_builder=model_builder,
            X=splits["X_full"],
            y=splits["y_full"],
            use_log_transform=True,
        )
        final_rows.append({
            "Dataset": dataset_name,
            "Model": model_name,
            **cv_summary,
        })
        print(f"  {model_name:16s} RMSE={cv_summary['RMSE_mean']:.2f} ± {cv_summary['RMSE_std']:.2f}")

    # 4. Reuse the fixed PSO hyperparameters for honest 5-fold CV reporting.
    print("\n[4/4] Running 5-fold CV for CNN variants...")
    baseline_cnn_summary = cross_validate_cnn(
        model_builder=make_cnn_builder(splits["n_features"]),
        X=splits["X_full"],
        y=splits["y_full"],
        batch_size=32,
        epochs=100,
        use_log_transform=True,
        verbose=0,
    )
    final_rows.append({
        "Dataset": dataset_name,
        "Model": "CNN_baseline",
        **baseline_cnn_summary,
    })
    print(f"  CNN_baseline    RMSE={baseline_cnn_summary['RMSE_mean']:.2f} ± {baseline_cnn_summary['RMSE_std']:.2f}")

    pso_cnn_summary = cross_validate_cnn(
        model_builder=make_cnn_builder(
            splits["n_features"],
            hyperparameters=best_hyperparameters,
        ),
        X=splits["X_full"],
        y=splits["y_full"],
        batch_size=int(best_hyperparameters.get("batch_size", 32)),
        epochs=100,
        use_log_transform=True,
        verbose=0,
    )
    final_rows.append({
        "Dataset": dataset_name,
        "Model": "CNN_PSO",
        **pso_cnn_summary,
    })
    print(f"  CNN_PSO         RMSE={pso_cnn_summary['RMSE_mean']:.2f} ± {pso_cnn_summary['RMSE_std']:.2f}")

print("\n\nAll datasets complete!")


  Dataset: china (499 samples, 18 features)

[1/2] Training baseline CNN...


2026-04-17 20:44:04,401 - pyswarms.single.global_best - INFO - Optimize for 25 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}


  Baseline CNN — MAE: 1103.52, RMSE: 4461.78, R²: 0.3377

[2/2] Running PSO (15 particles × 25 iterations)...


pyswarms.single.global_best: 100%|██████████|25/25, best_cost=0.316
2026-04-17 22:36:48,196 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.31575626134872437, best pos: [4.95399581e+01 3.65802519e+00 6.33170264e+01 8.51021474e-03
 2.21165177e-01 1.88874356e+00]


  Best PSO val MAE (log): 0.3158
  Hyperparams: {'filters': 50, 'kernel_size': 4, 'dense_units': 63, 'learning_rate': 0.008510214741517726, 'dropout_rate': 0.2211651774775311, 'num_conv_layers': 2}


...

  Baseline CNN — MAE: 322.19, RMSE: 642.34, R²: -0.3352

[2/2] Running PSO (15 particles × 25 iterations)...


pyswarms.single.global_best: 100%|██████████|25/25, best_cost=1.16
2026-04-18 01:00:24,854 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 1.1621530055999756, best pos: [7.48012653e+01 5.56902016e+00 1.32174230e+02 8.07267283e-03
 1.63265272e-01 1.91533191e+00]


  Best PSO val MAE (log): 1.1622
  Hyperparams: {'filters': 75, 'kernel_size': 6, 'dense_units': 132, 'learning_rate': 0.008072672826324971, 'dropout_rate': 0.16326527180867714, 'num_conv_layers': 2}


2026-04-18 01:00:36,139 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
2026-04-18 01:00:36,258 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 


  CNN+PSO    — MAE: 270.48, RMSE: 575.72, R²: -0.0726


All datasets complete!


In [ ]:
# --- Summary: leakage-free holdout results + final CV table ---

cnn_comparison = pd.concat(all_cnn_results, ignore_index=True)
cnn_cols = ["dataset", "model", "mae", "rmse", "r2", "mape", "mdmre", "pred25"]
cnn_comparison = cnn_comparison[cnn_cols].sort_values(["dataset", "rmse"]).reset_index(drop=True)

final_columns = [
    "Dataset",
    "Model",
    "RMSE_mean",
    "RMSE_std",
    "MAE_mean",
    "MAE_std",
    "R2_mean",
    "R2_std",
    "MAPE_mean",
    "MAPE_std",
    "MdMRE_mean",
    "MdMRE_std",
    "Pred25_mean",
    "Pred25_std",
]
full_comparison_final = pd.DataFrame(final_rows)
full_comparison_final = full_comparison_final[final_columns].sort_values(
    ["Dataset", "RMSE_mean", "Model"]
).reset_index(drop=True)

print("Leakage-free CNN holdout results")
print("=" * 70)
display(cnn_comparison)

print("\n5-fold cross-validation final comparison")
print("=" * 70)
full_comparison_final

CNN Baseline vs CNN+PSO (all datasets)


,dataset,model,mae,rmse,r2,mape,mdmre,pred25
0,china,cnn_baseline,1103.515147,4461.780604,0.337689,30.033954,0.232595,53.000000
1,china,cnn_pso,2047.838747,5577.414946,-0.034932,60.519170,0.470122,21.000000
5,cocomo81,cnn_pso,270.475983,575.724052,-0.072598,281.551205,0.928195,15.384615
4,cocomo81,cnn_baseline,322.193253,642.337642,-0.335165,98.698648,0.996362,0.000000
3,desharnais,cnn_pso,1592.690331,3476.033943,0.052979,34.708506,0.135443,64.705882
2,desharnais,cnn_baseline,4534.424807,5772.433553,-1.611618,99.929505,0.999554,0.000000


In [ ]:
# --- Save all outputs ---

# 1. Leakage-free holdout CNN comparison
save_metrics(cnn_comparison, file_name="cnn_vs_pso_metrics.csv", output_dir=metrics_dir)
print("Saved: cnn_vs_pso_metrics.csv")

# 2. Best PSO hyperparameters per dataset
hp_path = models_dir / "best_hyperparams.json"
with open(hp_path, "w") as file_handle:
    json.dump(all_hyperparams, file_handle, indent=2)
print(f"Saved: {hp_path}")

# 3. Training histories
hist_path = models_dir / "training_histories.json"
with open(hist_path, "w") as file_handle:
    json.dump(all_histories, file_handle, indent=2)
print(f"Saved: {hist_path}")

# 4. Standardized final 5-fold comparison
final_path = metrics_dir / "full_comparison_final.csv"
full_comparison_final.to_csv(final_path, index=False)
print(f"Saved: {final_path}")

# 5. Refresh the legacy combined comparison when baseline_metrics.csv exists
baseline_file = metrics_dir / "baseline_metrics.csv"
if baseline_file.exists():
    baselines = pd.read_csv(baseline_file)
    full_comparison = pd.concat([baselines, cnn_comparison], ignore_index=True)
    full_comparison = full_comparison.sort_values(["dataset", "rmse"])
    save_metrics(full_comparison, file_name="full_comparison.csv", output_dir=metrics_dir)
    print("Saved: full_comparison.csv")
else:
    print("baseline_metrics.csv not found; skipped legacy full_comparison.csv refresh")

Saved: cnn_vs_pso_metrics.csv
Saved: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\best_hyperparams.json
Saved: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\training_histories.json
Saved: full_comparison.csv

FULL COMPARISON — Best model per dataset (by RMSE)

  china: linear_regression (RMSE=1296.14, R²=0.9441)


,dataset,model,mae,rmse,r2,mape,mdmre,pred25
0,china,linear_regression,446.369674,1296.143319,0.944108,36.618911,0.112511,75.0
1,china,xgboost,388.871586,1503.300169,0.924814,11.363155,0.072925,91.0
2,china,random_forest,385.270333,1605.223998,0.914273,10.520690,0.078400,98.0
9,china,cnn_baseline,1103.515147,4461.780604,0.337689,30.033954,0.232595,53.0
10,china,cnn_pso,2047.838747,5577.414946,-0.034932,60.519170,0.470122,21.0



  cocomo81: random_forest (RMSE=404.22, R²=0.4712)


,dataset,model,mae,rmse,r2,mape,mdmre,pred25
3,cocomo81,random_forest,236.484410,404.224316,0.471247,198.270999,0.968000,15.384615
4,cocomo81,xgboost,278.135591,522.988085,0.114901,169.994311,0.774751,15.384615
11,cocomo81,cnn_pso,270.475983,575.724052,-0.072598,281.551205,0.928195,15.384615
12,cocomo81,cnn_baseline,322.193253,642.337642,-0.335165,98.698648,0.996362,0.000000
5,cocomo81,linear_regression,1490.513248,1921.078032,-10.942580,5166.806104,24.503235,7.692308



  desharnais: linear_regression (RMSE=1943.91, R²=0.7038)


,dataset,model,mae,rmse,r2,mape,mdmre,pred25
6,desharnais,linear_regression,1435.054687,1943.914123,0.703827,54.219786,0.296057,41.176471
7,desharnais,xgboost,1693.691995,2245.187760,0.604909,57.648496,0.404122,23.529412
8,desharnais,random_forest,1755.944118,2283.387200,0.591351,77.443114,0.343604,23.529412
13,desharnais,cnn_pso,1592.690331,3476.033943,0.052979,34.708506,0.135443,64.705882
14,desharnais,cnn_baseline,4534.424807,5772.433553,-1.611618,99.929505,0.999554,0.000000
